# Bringing It All Together

This notebook combines the three concepts you have learnt, distributions, confidence intervals, and hypothesis testing, to show how a complete analysis works from start to finish.

Each case study follows the same 7-step workflow you used in the exercise, this time fully worked with self-check answers along the way. The data cleaning is left out to keep the focus on the statistics.

## Learning Objectives

At the end of this notebook, you should be able to:

- Apply the 7-step workflow end to end on a business dataset.
- Check normality with histograms, Q-Q plots, and the Shapiro-Wilk test, and explain what each adds.
- Select between a paired t-test and a one-sample t-test from the structure of the data.
- Interpret test results as evidence for a business decision.

---
## Case Study 1: Effect of a Discount Promotion on Sales Volume

**Business context.** A local bookstore, "ReadMore", ran a discount promotion for one month and wants to evaluate its impact on sales volume. The store has collected the number of items sold per day before and during the promotion period.

The dataset contains the following columns:

| Column | Meaning |
|---|---|
| `Items_Sold_Before` | Items sold per day before the promotion |
| `Items_Sold_During` | Items sold per day during the promotion |

Each row is one day measured in both periods, so the two columns are **paired** observations. Keep that in mind for step 6.

In [ ]:
# The dataset

import pandas as pd

# Create DataFrame from the dataset
data = {
    "Items_Sold_Before": [40, 45, 50, 43, 42, 47, 48, 46, 44, 49],
    "Items_Sold_During": [55, 60, 70, 65, 68, 72, 75, 62, 69, 73],
}
df = pd.DataFrame(data)

### Step 1: Understand the Data Type

What data type are you working with?

Your answer here:


<details>
  <summary>Compare Answer</summary>
  <h3>Answer:</h3>
  <p>Items_Sold_Before and Items_Sold_During are continuous variables representing the number of items sold.
<p>
  
</details>

### Step 2: Identify the Analytical Question

Your answer here:


<details>
  <summary>Compare Answer</summary>
  <h3>Answer:</h3>
  <p>Question: Did the discount promotion lead to an increase in the number of items sold?
<p>
  
</details>




### Step 3: Calculate Descriptive Statistics

Before any test, look at the numbers. Descriptive statistics tell you whether there is anything worth testing and give you a first sense of the effect size.

In [ ]:
# Compute descriptive statistics
descriptive_stats = df.describe()
print(descriptive_stats)

On average the store sold about 45.4 items per day before the promotion and about 66.9 during it, an increase of more than 20 items per day. The spread also grows (standard deviation 3.2 before versus 6.3 during). The difference looks large; the hypothesis test in step 6 checks whether it is too large to be a fluke of these ten days.

### Step 4: Interpret the Graph and Descriptive Statistics

Plotting the two periods side by side shows where each distribution sits and how much they overlap.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.hist(df["Items_Sold_Before"], bins=6, alpha=0.7, color="blue")
plt.title("Items Sold Before Promotion")
plt.xlabel("Number of Items Sold")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
plt.hist(df["Items_Sold_During"], bins=6, alpha=0.7, color="green")
plt.title("Items Sold During Promotion")
plt.xlabel("Number of Items Sold")
plt.ylabel("Frequency")

plt.show()

### Interpretation

The two histograms barely overlap: every day during the promotion beats the best day before it. Neither histogram looks convincingly bell-shaped, but with only 10 observations per group a histogram is an unreliable guide to normality. Before reaching for a transformation or a non-parametric test, check normality properly, which is exactly what the next step does.

### Step 5 (Optional): Check for Normality with Q-Q Plots

#### What is a Q-Q plot?

A **Q-Q plot** is a scatter plot of the quantiles of your sample against the quantiles of a theoretical distribution. If your data follows that distribution, the points lie approximately along a straight line (typically y = x).

It is a visual check of distributional fit, and it is particularly useful for checking normality: more sensitive than a histogram at small sample sizes, because it does not depend on a bin width.

#### How to interpret a Q-Q plot

- **Points along the line**: the data follows the theoretical distribution. Systematic departures from the line indicate departures from that distribution.
- **S-shaped curve**: the tails of the data differ from the theoretical distribution, the data is more peaked or more spread out than a normal distribution.
- **Bulging or clustering**: points that drift away from the line in the middle or at the ends suggest skew or more extreme values than the theoretical distribution expects.

In [ ]:
import scipy.stats as stats

# Q-Q plot for items sold before promotion
stats.probplot(df["Items_Sold_Before"], dist="norm", plot=plt)
plt.title("Q-Q Plot Before Promotion")
plt.show()

# Q-Q plot for items sold during promotion
stats.probplot(df["Items_Sold_During"], dist="norm", plot=plt)
plt.title("Q-Q Plot During Promotion")
plt.show()

#### Interpretation

In both Q-Q plots the points lie close to the diagonal line, with no S-shape and no systematic drift at the ends. Both `Items_Sold_Before` and `Items_Sold_During` are therefore approximately normal, despite the lumpy histograms: this is why the Q-Q plot is worth the extra step at small sample sizes.

#### Verifying with the Shapiro-Wilk test

The **Shapiro-Wilk test** turns the visual check into a formal one. Its null hypothesis is that the data is normally distributed, so here a *large* p-value is good news: a p-value below 0.05 would indicate that the distribution is not normal.

In [ ]:
# Shapiro-Wilk test
# _ is a throwaway variable we are not really interested in this value
# but rather only the p-value
_, p_before = stats.shapiro(df["Items_Sold_Before"])
_, p_during = stats.shapiro(df["Items_Sold_During"])
print(f"Shapiro-Wilk p-value (Before): {p_before}")
print(f"Shapiro-Wilk p-value (During): {p_during}")

Both p-values are comfortably above 0.05 (about 0.97 before and 0.71 during), so the test finds no evidence against normality in either period. The data is normal or close to normal, so we need no transformation and no non-parametric test, and we can proceed with a t-test.

### Step 6: Formulate and Test Hypotheses

- Null hypothesis (H0): the average number of items sold before and during the promotion is the same.
- Alternative hypothesis (H1): the average number of items sold before and during the promotion is different.

Because each row is the same day measured in both periods, the right tool is a **paired t-test**: it tests whether the mean of the per-day differences is zero, removing day-to-day variation from the comparison.

The alternative: if the two columns were unrelated samples (say, ten random days from each of two different years), an **independent two-sample t-test** would be appropriate instead. Pairing only helps when the pairing is real.

In [ ]:
## Perform a paired t-test
t_stat, p_value = stats.ttest_rel(df["Items_Sold_Before"], df["Items_Sold_During"])
print(f"T-test Statistic: {t_stat}, p-value: {p_value}")

The paired t-statistic is about -14.5 with a p-value of roughly 1.5e-07, far below 0.05. A difference this consistent across all ten days is essentially impossible to explain by chance. Write your own conclusion before opening the answer below.

Your interpretation of the result:


<details>
  <summary>Compare Answer</summary>
  <h3>Answer:</h3>
  <p>Conclusion:
The results suggest that the discount promotion had a significant effect on increasing the number of items sold. Given the very small p-value, the observed increase in sales during the promotion is highly unlikely to be due to random chance alone.

<p>
  
</details>



---
## Case Study 2: Evaluating Average Customer Satisfaction Scores

**Business context.** A company recently launched a new product and surveyed customers to gauge satisfaction. The company's target satisfaction score for new products is 8.0 out of 10.

After collecting feedback from a sample of 15 customers, the company wants to determine whether the new product's average satisfaction score differs significantly from this target.

In [ ]:
import pandas as pd

# Create DataFrame
data = {
    "Satisfaction_Score": [
        7.5,
        8.1,
        7.8,
        8.3,
        8.0,
        7.9,
        8.4,
        7.6,
        8.2,
        8.1,
        7.7,
        8.5,
        7.4,
        8.0,
        8.3,
    ]
}
df = pd.DataFrame(data)

### Step 1: Understand the Data Type

Your answer here:




<details>
  <summary>Compare Answer</summary>
  <h3>Answer:</h3>
  <p>Satisfaction_Score is continuous and represents customer satisfaction on a 10-point scale.
</p>
  
</details>

### Step 2: Identify the Analytical Question

Your answer here:


<details>
  <summary>Compare Answer</summary>
  <h3>Answer:</h3>
  <p>Question: Is the average customer satisfaction score for the new product significantly different from the company’s target score of 8.0?</p>
  
</details>



### Step 3: Calculate Descriptive Statistics

In [ ]:
# Compute descriptive statistics
descriptive_stats = df.describe()
print(descriptive_stats)

The sample mean is about 7.99 with a standard deviation of about 0.33: on the face of it, almost exactly on the 8.0 target. The test in step 7 will tell us whether the tiny shortfall means anything.

### Step 4: Visualise the Distribution

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as stats

# Histogram
plt.hist(df["Satisfaction_Score"], bins=5, alpha=0.7, color="blue")
plt.title("Histogram of Satisfaction Scores")
plt.xlabel("Satisfaction Score")
plt.ylabel("Frequency")
plt.show()

### Step 5: Check Normality

The same two checks as in Case Study 1: a Q-Q plot for the visual read, then Shapiro-Wilk for the formal test.

In [ ]:
# Q-Q Plot
stats.probplot(df["Satisfaction_Score"], dist="norm", plot=plt)
plt.title("Q-Q Plot of Satisfaction Scores")
plt.show()

#### Confirm with the Shapiro-Wilk test

In [ ]:
# Shapiro-Wilk test for normality
_, p_value = stats.shapiro(df["Satisfaction_Score"])
print(f"Shapiro-Wilk p-value: {p_value}")

The Shapiro-Wilk p-value is about 0.82, well above 0.05, so there is no evidence against normality and a t-test is appropriate.

### Step 6: Formulate the Hypotheses

- Null hypothesis (H0): the average satisfaction score is equal to 8.0.
- Alternative hypothesis (H1): the average satisfaction score is not equal to 8.0.

### Step 7: Run the Test

Here we compare one sample mean with a fixed target value, so the tool is a **one-sample t-test**. We use a t-test rather than a z-test because the population standard deviation is unknown and must be estimated from the sample, exactly the situation the t-distribution was built for.

In [ ]:
# One-sample t-test
mu = 8.0
t_stat, p_value = stats.ttest_1samp(df["Satisfaction_Score"], mu)
print(f"T-test Statistic: {t_stat}, p-value: {p_value}")

Your interpretation here:


<details>
  <summary>Compare Answer</summary>
  <h3>Answer:</h3>
  <p>

P-Value Interpretation:

    
    A p-value of 0.879 is much greater than 0.05. 

    This indicates that there is no significant evidence to reject the null hypothesis.

    Therefore, you fail to reject the null hypothesis that the average customer satisfaction score is equal to 8.0.

Business Implications

Customer Satisfaction:

    The analysis suggests that the average satisfaction score of the new product is not significantly different from the target score of 8.0.

    From a business perspective, this means that the new product meets the company's satisfaction target based on this sample.

</p>
  
</details>

---
## Summary

In this notebook you:

- Applied the 7-step workflow end to end on two business datasets.
- Checked normality three ways: histograms for a first look, Q-Q plots for a reliable visual read at small sample sizes, and Shapiro-Wilk for a formal test.
- Selected a paired t-test for before/during measurements of the same days, and a one-sample t-test for comparing a mean with a fixed target.
- Interpreted one emphatic rejection (promotion effect, p about 1.5e-07) and one clear non-rejection (satisfaction on target, p about 0.88) as business conclusions.

| Case study | Test | p-value | Decision at alpha = 0.05 |
|---|---|---|---|
| Promotion effect on sales | Paired t-test | 1.5e-07 | Reject H0: sales increased |
| Satisfaction vs 8.0 target | One-sample t-test | 0.88 | Fail to reject H0: on target |

## References & Further Reading

- [**scipy.stats.ttest_rel**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html): The paired t-test.
- [**scipy.stats.ttest_1samp**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_1samp.html): The one-sample t-test.
- [**scipy.stats.shapiro**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html): The Shapiro-Wilk normality test.
- [**scipy.stats.probplot**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.probplot.html): The function behind the Q-Q plots.
- [**ModernDive: Hypothesis Testing**](https://moderndive.com/v2/hypothesis-testing.html): A simulation-first treatment of the same workflow.